# Delta Lake Incremental Data Processing Assignment

This notebook demonstrates incremental data processing using Delta Lake MERGE operations with strict schemas, validations, and SCD Type 1 & 2 support.

In [1]:
# Setup PySpark with Delta Lake
import os
import sys
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType
from pyspark.sql.functions import col, lit, current_date, when
from delta.tables import DeltaTable

# Cross-platform Hadoop environment setup for Windows
if sys.platform.startswith("win"):
    hadoop_dir = os.environ.get("HADOOP_HOME", "C:\\hadoop")
    if os.path.exists(hadoop_dir):
        os.environ["HADOOP_HOME"] = hadoop_dir
        bin_dir = os.path.join(hadoop_dir, "bin")
        if os.path.exists(bin_dir) and bin_dir not in os.environ.get("PATH", ""):
            os.environ["PATH"] = bin_dir + os.path.pathsep + os.environ.get("PATH", "")

# Configure Spark
builder = SparkSession.builder.appName("DeltaLakeAssignmentEnterprise") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")

spark = configure_spark_with_delta_pip(builder).getOrCreate()

# Dynamically resolve project directory structure across any OS and runner
base_dir = os.path.dirname(os.path.abspath(__file__)) if '__file__' in globals() else os.getcwd()
project_root = os.path.dirname(base_dir) if os.path.basename(base_dir) == 'notebooks' else base_dir

data_dir = os.path.join(project_root, 'data')
master_csv_path = os.path.join(data_dir, 'customer_master.csv')
incremental_csv_path = os.path.join(data_dir, 'customer_incremental.csv')
delta_scd1_path = os.path.join(data_dir, 'delta_scd1')
delta_scd2_path = os.path.join(data_dir, 'delta_scd2')

print(f"Spark setup complete. Working with project root: {project_root}")


Spark setup complete. Working with project root: c:\Users\Spandan Swarup Nanda\Desktop\week 7 celebal\delta-lake-assignment


## 1. Explicit Schema Definition & Data Loading

In [2]:
# Define strict schema to prevent schema drift
customer_schema = StructType([
    StructField("customer_id", StringType(), False),
    StructField("customer_name", StringType(), True),
    StructField("segment", StringType(), True),
    StructField("country", StringType(), True),
    StructField("city", StringType(), True),
    StructField("state", StringType(), True),
    StructField("postal_code", IntegerType(), True),
    StructField("region", StringType(), True)
])

# Load customer master data using the explicit schema
master_df = spark.read.csv(master_csv_path, header=True, schema=customer_schema)

master_df.write.format("delta").mode("overwrite").save(delta_scd1_path)

# For SCD2, add tracking columns
scd2_initial_df = master_df \
    .withColumn("is_active", lit(True)) \
    .withColumn("effective_date", current_date()) \
    .withColumn("end_date", lit(None).cast(DateType()))

scd2_initial_df.write.format("delta").mode("overwrite").save(delta_scd2_path)

print("Data loading completed securely.")


Data loading completed securely.


## 2. Robust Data Cleaning (with DQ Assertions)

In [3]:
deltaTable_scd1 = DeltaTable.forPath(spark, delta_scd1_path)
df_clean = deltaTable_scd1.toDF().dropDuplicates(["customer_id"]).dropna(subset=["customer_id"])
df_clean.write.format("delta").mode("overwrite").save(delta_scd1_path)

deltaTable_scd2 = DeltaTable.forPath(spark, delta_scd2_path)
df_clean_scd2 = deltaTable_scd2.toDF().dropDuplicates(["customer_id"]).dropna(subset=["customer_id"])
df_clean_scd2.write.format("delta").mode("overwrite").save(delta_scd2_path)

# Data Quality Assertions
cleaned_count = df_clean.count()
null_ids = df_clean.filter(col("customer_id").isNull()).count()
assert null_ids == 0, "Data Quality Error: Null Customer IDs found!"
print(f"Data cleaned successfully. Valid records: {cleaned_count}")


Data cleaned successfully. Valid records: 400


## 3. Load Incremental Data

In [4]:
incremental_df = spark.read.csv(incremental_csv_path, header=True, schema=customer_schema)
incremental_count = incremental_df.count()
print(f"Loaded {incremental_count} incremental records.")


Loaded 70 incremental records.


## 4. SCD Type 1 Operation (MERGE Overwrite)

In [5]:
deltaTable_scd1 = DeltaTable.forPath(spark, delta_scd1_path)

deltaTable_scd1.alias("target").merge(
    incremental_df.alias("source"),
    "target.`customer_id` = source.`customer_id`"
) \
  .whenMatchedUpdateAll() \
  .whenNotMatchedInsertAll() \
  .execute()

final_scd1_df = spark.read.format("delta").load(delta_scd1_path)
print(f"SCD1 MERGE completed. Total rows: {final_scd1_df.count()}")


SCD1 MERGE completed. Total rows: 450


## 5. SCD Type 2 Operation (Maintain History)

In [6]:
deltaTable_scd2 = DeltaTable.forPath(spark, delta_scd2_path)

# SCD2 Logic requires updates for existing, inserts for existing (new version), and inserts for new records
# 1. Prepare incremental data with SCD2 flags
updates_df = incremental_df \
    .withColumn("is_active", lit(True)) \
    .withColumn("effective_date", current_date()) \
    .withColumn("end_date", lit(None).cast(DateType()))

# 2. Identify records that are being updated (exist in both, and active in target)
staged_updates = updates_df.alias("updates") \
    .join(deltaTable_scd2.toDF().alias("target"), "customer_id") \
    .where("target.is_active = true") \
    .selectExpr("NULL as mergeKey", "updates.*")

# 3. New records and new versions of existing records
staged_inserts = updates_df.selectExpr("`customer_id` as mergeKey", "*")

# Combine staged data
staged_df = staged_updates.unionByName(staged_inserts)

# 4. Perform SCD2 MERGE
deltaTable_scd2.alias("target").merge(
    staged_df.alias("source"),
    "target.`customer_id` = source.mergeKey"
) \
  .whenMatchedUpdate(
      condition = "target.is_active = true",
      set = {
          "is_active": "false",
          "end_date": "current_date()"
      }
  ) \
  .whenNotMatchedInsertAll() \
  .execute()

final_scd2_df = spark.read.format("delta").load(delta_scd2_path)
print(f"SCD2 MERGE completed. Total rows (including history): {final_scd2_df.count()}")


SCD2 MERGE completed. Total rows (including history): 470


## 6. Validation and Output

In [7]:
# Assertions
assert final_scd1_df.count() > 0, "SCD1 output is empty"
assert final_scd2_df.count() > final_scd1_df.count(), "SCD2 should have more rows than SCD1 due to history tracking"

print("All DQ Assertions passed successfully!")
print("\n--- SCD Type 1 Final State (Top 5) ---")
final_scd1_df.filter(col("city") == 'New Enterprise City').show(5, truncate=False)

print("\n--- SCD Type 2 Historical View (Top 5 Updates) ---")
final_scd2_df.filter(col("city") == 'New Enterprise City').orderBy("customer_id", "effective_date").show(5, truncate=False)


All DQ Assertions passed successfully!

--- SCD Type 1 Final State (Top 5) ---
+-----------+---------------+---------+-------------+-------------------+--------------------+-----------+------+
|customer_id|customer_name  |segment  |country      |city               |state               |postal_code|region|
+-----------+---------------+---------+-------------+-------------------+--------------------+-----------+------+
|AA-10480   |Andrew Allen   |Consumer |United States|New Enterprise City|New Enterprise State|28027      |South |
|AG-10270   |Alejandro Grove|Consumer |United States|New Enterprise City|New Enterprise State|84084      |West  |
|BH-11710   |Brosina Hoffman|Consumer |United States|New Enterprise City|New Enterprise State|90032      |West  |
|CG-12520   |Claire Gute    |Consumer |United States|New Enterprise City|New Enterprise State|42420      |South |
|DV-13045   |Darrin Van Huff|Corporate|United States|New Enterprise City|New Enterprise State|90036      |West  |
+--------